# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
from functools import reduce

In [0]:
%run ./src_text_cleaning

# Reading from Bronze

In [0]:
df_manifest = spark.table("humanoid_robotics.bronze.manifest_pilot")

df_docs = spark.table("humanoid_robotics.bronze.documents")

# Data Transformations

## Join manifest_pilot with documents

In [0]:
df = (
    df_manifest
    .join(df_docs, on = "file_name", how = "left")
)
df.display()

### Translate non-English documents

In [0]:
is_zh = F.coalesce((F.col("language") == "zh") & F.col("raw_text").isNotNull(), F.lit(False))

df_zh = (df.filter(is_zh)
           .withColumn("title", F.expr("ai_translate(title, 'en')"))
           .withColumn("publisher", F.expr("ai_translate(publisher, 'en')"))
           .withColumn("raw_text", F.expr("ai_translate(raw_text, 'en')"))
           .withColumn("is_translated", F.lit(True)))

df_other = (df.filter(~is_zh | F.col("language").isNull())
              .withColumn("is_translated", F.lit(False)))


df = (
    df_zh.unionByName(df_other)
    .withColumn("publish_date", F.to_date("publish_date"))
    .withColumn("char_count", F.length("raw_text"))
    .select("doc_id", "title", "publisher", "source_type", "language", "publish_date", "primary_company", "raw_text", "char_count", "url")
)



# Write into Silver Table

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("humanoid_robotics.silver.documents"))

df = spark.table("humanoid_robotics.silver.documents")
display(df)

## Data Inspection 

In [0]:
total     = df.count()
missing   = df.filter(F.col("raw_text").isNull()).count()
too_short = df.filter(F.col("char_count") < 300).count()
dup_ids   = total - df.select("doc_id").distinct().count()

print(f"total_rows: {total}")
print(f"null_text: {missing}")
print(f"texts<300: {too_short}")
print(f"dup_doc_id: {dup_ids}")

display(df.filter("language = 'zh'").select("doc_id", F.substring("raw_text", 1, 200).alias("preview")))

## Rule_based text cleaning

In [0]:
df_docs = df.filter(F.col("raw_text").isNotNull()).select("doc_id", "raw_text", "language")

df_lines = split_into_lines(df_docs)

df_flagged = flag_lines(df_lines)

df_clean   = build_documents_clean(df_docs, df_flagged)

In [0]:
display(df_clean.orderBy(F.desc("removed_ratio")))

### Delete Log

In [0]:
display(
    df_flagged.filter("removed_reason IS NOT NULL")
              .groupBy("removed_reason").count()
              .orderBy(F.desc("count"))
)

display(
    df_flagged.filter("removed_reason IS NOT NULL")
              .orderBy("doc_id", "line_no")
)

## Write into Silver Tables

In [0]:
(df_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")  
    .saveAsTable("humanoid_robotics.silver.documents_clean"))